In [ ]:
# Configurar el motor de Keras antes de importarlo
import os
os.environ["KERAS_BACKEND"] = "torch"

# Librerías generales
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch

# Herramientas de scikit-learn
from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler

# Construcción de la red con Keras
from keras import Input
from keras.models import Sequential, clone_model
from keras.layers import Dense, LeakyReLU, Dropout
from keras.regularizers import L1L2
from keras.callbacks import EarlyStopping
from keras.metrics import RootMeanSquaredError
from keras.optimizers import SGD, RMSprop

# Opciones de visualización de tablas
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

# Comprobar el acceso a la GPU
print("GPU disponible:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("Placa:", torch.cuda.get_device_name(0))

In [ ]:
datos_train=pd.read_csv(r"C:\Users\juane\OneDrive\Desktop\Redes FeedForward\regresion_train\regresion_train.csv")
datos_test=pd.read_csv(r"C:\Users\juane\OneDrive\Desktop\Redes FeedForward\regresion_train\regresion_test_features.csv")
datos_train.head()

In [ ]:
def miModelo(datos_train, datos_test, random_state=42, scale=True, test_size=0.05):
    
    X = datos_train.iloc[:, :-1].to_numpy(dtype=np.float32)
    y = datos_train.iloc[:, -1].to_numpy(dtype=np.float32).reshape(-1, 1)

    X_t = datos_test.to_numpy(dtype=np.float32)
    
    X_train, X_validacion, y_train, y_validacion = train_test_split(X, y, test_size=test_size, random_state=random_state, shuffle=True)

    scaler = StandardScaler()

    X_train = scaler.fit_transform(X_train)
    X_validacion = scaler.transform(X_validacion)
    X_t = scaler.transform(X_t)

    modelo = Sequential([Input(shape=(X_train.shape[1],))])

    for _ in range(4):
        modelo.add(Dense(64, activation="leaky_relu", kernel_regularizer=L1L2(l1=1e-4, l2=0.0)))
        modelo.add(Dropout(0.225))

    modelo.add(Dense(1, activation="linear"))
    
    opt = SGD(learning_rate=0.00001)

    modelo.compile(loss="mean_squared_error", optimizer=opt, metrics=["mae", RootMeanSquaredError(name="rmse")])

    return (modelo, X_train, X_validacion, X_t, y_train, y_validacion)

In [ ]:
def entrenamientoModelo(modelo, X_train, y_train, X_validacion, y_validacion, iteracion, epochs=1000, patience=80):
    
    parada = EarlyStopping(monitor="val_rmse", mode="min", patience=patience, min_delta=0.0000001, restore_best_weights=True, verbose=1)

    resultado = modelo.fit(X_train, y_train, validation_data=(X_validacion, y_validacion), epochs=epochs, batch_size=32, shuffle=True, callbacks=[parada], verbose=1)

    metricas = modelo.evaluate(X_validacion, y_validacion, return_dict=True, verbose=0)

    print(f"\nIteración {iteracion} — " f"RMSE: {metricas['rmse']:.4f} — " f"MAE: {metricas['mae']:.4f}")

    epocas = np.arange(1, len(resultado.history["loss"]) + 1)

    fig, ejes = plt.subplots(1, 3, figsize=(18, 4))

    curvas = [("loss", "Loss: MSE + regularización"), ("mae", "MAE"), ("rmse", "RMSE")]

    for eje, (metrica, titulo) in zip(ejes, curvas):

        eje.plot(epocas, resultado.history[metrica], label="Train")
        eje.plot(epocas, resultado.history[f"val_{metrica}"], label="Validación")

        eje.set_title(titulo)
        eje.set_xlabel("Época")
        eje.set_ylabel(titulo)
        eje.grid(alpha=0.3)
        eje.legend()

    fig.suptitle(f"Iteración {iteracion}")
    plt.tight_layout()
    plt.show()

    return resultado, metricas

In [ ]:
def testeo(modelo, X_t):
    predictions = modelo.predict(X_t)

    predicted_value = np.array(predictions)
    predicted_value = predicted_value.reshape(max(predicted_value.shape), )

    return predicted_value


In [ ]:
def graficar_train_test(predicciones_test, target_train):
    
    target_train = np.asarray(target_train).ravel()
    predicciones_test = np.asarray(predicciones_test).ravel()

    n_train = len(target_train)
    n_test = len(predicciones_test)

    x_train = np.arange(n_train)
    x_test = np.arange(n_train, n_train + n_test)

    plt.figure(figsize=(10, 5))

    plt.scatter(x_train, target_train,color="tab:blue", label="Train: target real", alpha=0.7)

    plt.scatter(x_test, predicciones_test, color="tab:orange", label="Test: predicción", marker="x")

    plt.xlabel("Índice de muestra (primero train, después test)")
    plt.ylabel("Valor del target / predicción")
    plt.legend()
    plt.grid(alpha=0.25)
    plt.tight_layout()
    plt.show()

In [ ]:
modelo, X_train, X_validacion, X_t, y_train, y_validacion = miModelo(datos_train, datos_test, random_state=42, test_size=0.05)

historia, metricas = entrenamientoModelo(modelo, X_train, y_train, X_validacion, y_validacion, iteracion=1, epochs=1000, patience=120)

print(f"\nRMSE de validación: {metricas['rmse']:.4f}")
print(f"MAE de validación: {metricas['mae']:.4f}")

modelo.save("modelo_regresion.keras")

predicciones = modelo.predict(X_t, verbose=0).ravel()


In [ ]:
pd.DataFrame({"y_pred": predicciones}).to_csv("entrega_regresion.csv", index=False)

In [ ]:
def chequear(path, filas, etiquetas=None):
    df = pd.read_csv(path)
    assert list(df.columns) == ["y_pred"], f"Columnas: {list(df.columns)}"
    assert len(df) == filas, f"Hay {len(df)} filas y se esperan {filas}"
    if etiquetas:
        malas = set(df.y_pred) - set(etiquetas)
        assert not malas, f"Etiquetas inválidas: {malas}"
    else:
        assert df.y_pred.notna().all(), "Hay valores vacíos"
    print("Formato OK")

chequear("entrega_regresion.csv", 200)

In [ ]:

modelo.save(f"modelo_rmse3.0324.keras")